# Full Atlas-Free CNN Brain-Text Pipeline

Official end-to-end runner. Reusable training/evaluation logic lives in `atlas_free_cnn` modules and `experiments/3dcnn/train_ale_cnn.py`; this notebook only sets config switches, creates structured output folders, and launches requested stages.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match the working Colab setup used by Notebook 2:
# code repo in /content/neurovlm_gnn, Drive used for data and run outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
            "or remove that folder, then rerun this cell."
        )
    run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    if checkout.returncode != 0:
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", os.getcwd())
print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

from atlas_free_cnn.pipeline_outputs import (
    create_full_pipeline_run_dir,
    git_info,
    write_json,
    write_status_report,
    write_readme_what_to_look_at,
    write_table,
    flatten_stage3_metrics,
    ae_source_metric_columns,
)
from atlas_free_cnn.training.train_autoencoder import train_from_config, train_stage1b_from_config


## Config Switches

Keep `AE_TRAINING_RECIPE='baseline_raw_mse'` for the required raw-MSE baseline. The legacy name `previous_good_compatible` is still accepted as an alias. Improved AE runs must be selected explicitly.

In [ ]:
def split_dir_has_jsonl(path: Path) -> bool:
    return all((path / name).exists() for name in ["train.jsonl", "val.jsonl", "test.jsonl"])


def discover_unified_split_dir() -> Path:
    override = os.environ.get("NEUROVLM_UNIFIED_SPLIT_DIR")
    candidates = []
    if override:
        candidates.append(Path(override))
    candidates.extend([
        REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "cache/unified_jsonl/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/unified_jsonl/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "data_ale_3dcnn/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "data_ale_3dcnn/unified_jsonl/splits",
    ])
    for candidate in candidates:
        if split_dir_has_jsonl(candidate):
            return candidate
    checked = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not find unified dataset split JSONL files. Expected train.jsonl, val.jsonl, and test.jsonl in one of:\n"
        f"{checked}\n\n"
        "If your splits are elsewhere, set os.environ['NEUROVLM_UNIFIED_SPLIT_DIR'] to that splits directory before running this cell. "
        "The directory must contain train.jsonl, val.jsonl, and test.jsonl, and the tensor_path/nifti_path values inside them must point to files accessible from Colab/Drive."
    )

UNIFIED_SPLIT_DIR = discover_unified_split_dir()
TRAIN_JSONL = str(UNIFIED_SPLIT_DIR / "train.jsonl")
VAL_JSONL = str(UNIFIED_SPLIT_DIR / "val.jsonl")
TEST_JSONL = str(UNIFIED_SPLIT_DIR / "test.jsonl")
print("Unified split dir:", UNIFIED_SPLIT_DIR)
print("Train JSONL:", TRAIN_JSONL)

RUN_MODE = "full"  # full | ae_only | downstream_only
DATA_MODE = "mixed"  # mixed | pubmed_only | statmaps_only
AE_TRAINING_RECIPE = "baseline_raw_mse"  # baseline_raw_mse | mixed_balanced_raw_mse | mixed_balanced_hybrid_loss
AE_VARIANT = "mixed_baseline_raw_mse"
AE_CHECKPOINT_SELECTION = "best_val_loss"  # best_val_loss | best_spatial_corr | best_top1_dice | best_top5_dice | best_foreground_mse | last
AE_INIT_VARIANT = "mixed_baseline"  # mixed_baseline | mixed_balanced | mixed_hybrid | mixed_to_pubmed | mixed_to_statmaps | custom path
AE_CKPT_PATH = None  # explicit checkpoint path overrides variant/selection

RUN_STAGE1_AUTOENCODER = True
RUN_STAGE1B_FINETUNE = False
STAGE1B_MODE = "mixed_pretrain_to_pubmed"  # mixed_pretrain_to_pubmed | mixed_pretrain_to_statmaps
RUN_STAGE3_CONTRASTIVE = True
RUN_STAGE4_TEXT_TO_BRAIN = False
RUN_STAGE5_GENERATION_EVAL = False

BASE_OUTPUT_DIR = DRIVE_ROOT / "runs_atlas_free_cnn_full_pipeline" if "DRIVE_ROOT" in globals() else Path("runs_atlas_free_cnn_full_pipeline")


In [ ]:
paths = create_full_pipeline_run_dir(BASE_OUTPUT_DIR)
RUN_DIR = Path(paths["run_dir"])
print(f"Run directory: {RUN_DIR}")

metadata_dir = Path(paths["metadata"])
write_json(metadata_dir / "run_config.json", {
    "RUN_MODE": RUN_MODE,
    "DATA_MODE": DATA_MODE,
    "AE_TRAINING_RECIPE": AE_TRAINING_RECIPE,
    "AE_VARIANT": AE_VARIANT,
    "AE_CHECKPOINT_SELECTION": AE_CHECKPOINT_SELECTION,
    "AE_INIT_VARIANT": AE_INIT_VARIANT,
    "AE_CKPT_PATH": AE_CKPT_PATH,
    "RUN_STAGE1_AUTOENCODER": RUN_STAGE1_AUTOENCODER,
    "RUN_STAGE1B_FINETUNE": RUN_STAGE1B_FINETUNE,
    "RUN_STAGE3_CONTRASTIVE": RUN_STAGE3_CONTRASTIVE,
    "RUN_STAGE4_TEXT_TO_BRAIN": RUN_STAGE4_TEXT_TO_BRAIN,
    "RUN_STAGE5_GENERATION_EVAL": RUN_STAGE5_GENERATION_EVAL,
})
write_json(metadata_dir / "git_info.json", git_info(REPO_DIR))
(metadata_dir / "environment.txt").write_text(sys.version)

## Stage 1: AE Pretraining

In [ ]:
selected_ae_checkpoint = AE_CKPT_PATH
stage1_result = None
if RUN_STAGE1_AUTOENCODER:
    stage1_dir = Path(paths["stage1"])
    ae_cfg = {
        "train_jsonl": TRAIN_JSONL,
        "val_jsonl": VAL_JSONL,
        "test_jsonl": TEST_JSONL,
        "output_dir": str(stage1_dir),
        "checkpoint_dir": str(stage1_dir / "checkpoints"),
        "data_mode": DATA_MODE,
        "ae_training_recipe": AE_TRAINING_RECIPE,
        "ae_variant": AE_VARIANT,
        "checkpoint_selection_metric": AE_CHECKPOINT_SELECTION,
        "target_shape": [36, 45, 38],
        "model": {"latent_dim": 384, "base_channels": 64, "num_blocks": 4, "dropout": 0.1, "norm": "group", "pooling": "max"},
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "amp": True,
        "gradient_clipping": 1.0,
        "batch_size": 64,
        "epochs": 300,
        "source_sampling": "natural" if AE_TRAINING_RECIPE in {"baseline_raw_mse", "previous_good_compatible"} else "balanced",
        "loss": {"type": "raw_mse", "lambda_foreground": 0.0, "lambda_topk": 0.0, "prediction_activation": "none"},
    }
    if AE_TRAINING_RECIPE == "mixed_balanced_hybrid_loss":
        ae_cfg["loss"] = {"type": "hybrid_recon", "lambda_foreground": 0.10, "lambda_topk": 0.05, "topk_percent": 5, "prediction_activation": "none"}
    stage1_result = train_from_config(ae_cfg)
    selected_ae_checkpoint = str(Path(stage1_result["checkpoint_dir"]) / (AE_CHECKPOINT_SELECTION + ".pt" if AE_CHECKPOINT_SELECTION == "last" else f"{AE_CHECKPOINT_SELECTION}.pt"))
    if not Path(selected_ae_checkpoint).exists():
        selected_ae_checkpoint = stage1_result["best_checkpoint"]
else:
    print("Stage 1 not requested")
print("selected_ae_checkpoint=", selected_ae_checkpoint)

## Stage 1B: Optional Domain Fine-Tuning

In [ ]:
stage1b_result = None
if RUN_STAGE1B_FINETUNE:
    domain = "pubmed" if STAGE1B_MODE == "mixed_pretrain_to_pubmed" else "statmaps"
    stage1b_dir = Path(paths["stage1b"]) / domain
    ft_cfg = {
        "stage1b_mode": STAGE1B_MODE,
        "mixed_pretrain_checkpoint": selected_ae_checkpoint,
        "train_jsonl": TRAIN_JSONL,
        "val_jsonl": VAL_JSONL,
        "test_jsonl": TEST_JSONL,
        "output_dir": str(stage1b_dir),
        "checkpoint_dir": str(stage1b_dir / "checkpoints"),
        "ae_training_recipe": "baseline_raw_mse",
        "ae_variant": "mixed_to_pubmed" if domain == "pubmed" else "mixed_to_statmaps",
        "checkpoint_selection_metric": AE_CHECKPOINT_SELECTION,
        "lr": 1e-4,
        "freeze_mode": "none",
        "epochs": 100,
        "loss": {"type": "raw_mse", "prediction_activation": "none"},
    }
    stage1b_result = train_stage1b_from_config(ft_cfg)
    selected_ae_checkpoint = stage1b_result["best_checkpoint"]
else:
    print("Stage 1B not requested")

## Stage 2/3: Encoder Initialization and Contrastive Training

In [ ]:
if RUN_STAGE3_CONTRASTIVE:
    stage3_dir = Path(paths["stage3"])
    ckpt_dir = stage3_dir / "checkpoints"
    cmd = [
        sys.executable, "experiments/3dcnn/train_ale_cnn.py",
        "--mode", "atlas_free",
        "--model", "ale_3dcnn",
        "--encoder-init", "autoencoder_pretrained",
        "--ae-ckpt-path", str(selected_ae_checkpoint),
        "--ae-init-variant", str(AE_INIT_VARIANT),
        "--ae-checkpoint-selection", AE_CHECKPOINT_SELECTION,
        "--text-proj-init", "pretrained_infonce",
        "--run-dir", str(stage3_dir),
        "--checkpoint-dir", str(ckpt_dir),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Stage 3 not requested")

## Stage 4/5 Hooks

Stage 4 and Stage 5 remain script-driven. Enable these cells once the desired text-to-brain and generation-eval config is set for this run directory.

In [ ]:
if RUN_STAGE4_TEXT_TO_BRAIN:
    print("Configure and launch atlas_free_cnn.training.train_text_to_brain here; outputs should go under", Path(paths["stage4"]))
else:
    print("Stage 4 not requested")

if RUN_STAGE5_GENERATION_EVAL:
    print("Configure and launch generation evaluation here; outputs should go under", Path(paths["stage5"]))
else:
    print("Stage 5 not requested")

## Final Status and Comparison Files

In [ ]:
stage_status = write_status_report(RUN_DIR, {
    "stage1": RUN_STAGE1_AUTOENCODER,
    "stage1b": RUN_STAGE1B_FINETUNE,
    "stage3": RUN_STAGE3_CONTRASTIVE,
    "stage4": RUN_STAGE4_TEXT_TO_BRAIN,
    "stage5": RUN_STAGE5_GENERATION_EVAL,
})

row = {
    "ae_variant": AE_VARIANT,
    "ae_checkpoint_selection": AE_CHECKPOINT_SELECTION,
    "ae_checkpoint_path": selected_ae_checkpoint or "",
    "ae_epoch": "",
    "stage5_generation_spatial_corr": "",
    "stage5_generation_top5_dice": "",
    "notes": "Generated by 5_full_atlas_free_cnn_pipeline.ipynb",
}
ae_metrics_path = Path(paths["stage1"]) / "metrics" / "reconstruction_summary_by_source.csv"
row.update(ae_source_metric_columns(ae_metrics_path))
if selected_ae_checkpoint and Path(selected_ae_checkpoint).exists():
    try:
        import torch
        row["ae_epoch"] = torch.load(selected_ae_checkpoint, map_location="cpu", weights_only=False).get("epoch", "")
    except Exception:
        pass
metrics_path = Path(paths["stage3"]) / "metrics" / "test_metrics.json"
if metrics_path.exists():
    row.update(flatten_stage3_metrics(json.loads(metrics_path.read_text())))
write_table(Path(paths["final"]) / "ae_to_downstream_comparison.csv", [row])
write_table(Path(paths["final"]) / "final_summary_table.csv", [{"stage": s["stage"], "status": s["status"]} for s in stage_status])
write_table(Path(paths["final"]) / "best_checkpoints_to_inspect.csv", [{"name": "selected_ae", "path": selected_ae_checkpoint or ""}])
write_readme_what_to_look_at(
    Path(paths["final"]) / "README_WHAT_TO_LOOK_AT.md",
    ae_variant=AE_VARIANT,
    ae_selection=AE_CHECKPOINT_SELECTION,
    ae_checkpoint_path=selected_ae_checkpoint or "",
    stage_status=stage_status,
)
for s in stage_status:
    print(f"{s['stage']}: {s['status']}")
print("Final comparison:", Path(paths["final"]))